# 1.概述

## 1.1 上下文工程

在 LangChain中，记忆（Memory）专门负责 **存储示例交互信息** 的组件，核心作用是 **保存上下文** 和 **提供上下文**，让LLM在每次响应时，都能看到之前的对话内容。

**上下文工程（Context Engineering）** 负责合理组织这些这些记忆和任务信息，让LLM的响应更连贯、更切合需求。这也是Agent能实现复杂多轮交互的核心。

## 1.2 上下文类型与相关API
LangChain 的上下文工程是基于Agent讨论的，构建在 LangGraph 之上。

LangGraph 提供了三种管理上下文的方法，这些方法结合了可变性和生命周期维度：

| 上下文类型 | 描述 | 可变性 | 生命周期 | 访问方法 |
| --- | --- | --- | --- | --- |
| 动态运行时上下文 | 在单次运行中会演变的可变数据 | 动态 | 单次运行 | LangGraph state 对象 |
| 动态跨会话上下文 | 在对话间共享的持久数据。比如用户偏好、历史洞察、知识条目 | 动态 | 跨对话 | LangGraph store 对象 |
| 静态运行时上下文 | 在启动时传入的用户元数据、工具、数据库连接 | 静态 | 单次运行 | LangGraph context 对象 |

## 1.3 LangChain的记忆

### 1.3.1 记忆的分类

官方说明: https://docs.langchain.com/oss/python/concepts/memory

记忆分为短期记忆和长期记忆，对应不同的使用场景:

- **短期记忆**（Short-term memory、会话级记忆、thread-scoped memory）：作用范围是单个对话线程（Thread）内，一旦开启新对话（更换 `thread_id`），记忆即消失。
- **长期记忆**（Long-term memory，跨会话级记忆）：会在会话间存储用户特定或应用级数据，并在会话线程间共享。它可以随时在任何线程中被调用。记忆的范围是任意自定义命名空间，而不仅仅是单一线程 ID。

### 1.3.2 记忆的管理

LangChain v1.x版本中，Agent是构建在LangGraph图结构之上的，通过上文提到的state和store构建记忆系统。

- **state**: **短期记忆对象**，以会话为单位组织，包含当前会话的所有消息记录以及自定义信息。
- **store**: **长期记忆对象**，跨会话持久化的数据，通常需要结合向量数据库或外部存储实现。

# 2. 短期记忆

LangChain 1.x 的短期记忆是三者的组合：

State (会话内部状态) + Checkpointer (持久化机制) + Thread ID (会话作用域)

- **State**：默认存储历史消息列表 messages，通过 State 管理历史消息
- **Checkpointer**：负责将 State 作为检查点持久化保存，检查点是某个时刻的 State 快照
- **Thread ID**：用于唯一标识 State，LangChain 运行时会按照 thread_id 读写 State 快照

**State 和 Checkpointer 的关系:**
- State 是数据，Checkpointer 是存取机制。
- State：Agent 运行时的内部状态（默认是 messages 历史消息列表，可自定义字段）。存在于内存中，进程结束即丢。
- Checkpointer：把 State 在某个时刻的快照写入外部存储（内存/PostgreSQL/SQLite）的组件，负责"存"和"恢复"。
- Thread ID：给每份 State 命名。运行时会按 thread_id 去 checkpointer 里读写对应快照。
- 关系一句话：invoke 时 agent 从 checkpointer 按 thread_id 读出历史 State 恢复上下文 → 运行中新消息更新 State → 结束时把新 State 快照存回 checkpointer。所以同一个 thread_id 的多次调用是连续对话，换 thread_id 则从零开始（见 2.6 notebook 第 1 节）。

## 2.1 基于内存的持久化器

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}

agent = create_agent(model=model, checkpointer=checkpointer)

resp = agent.invoke(
    {"messages": [HumanMessage("你好，我叫aihaipeng，喜欢仓鼠")]}, config=config
)

for msg in resp["messages"]:
    msg.pretty_print()

In [ ]:
resp = agent.invoke(
    {"messages": [HumanMessage("还记的我的名字和喜欢的小动物么？")]}, config=config
)
for msg in resp["messages"]:
    msg.pretty_print()

In [ ]:
from rich import print as rprint

state = agent.get_state(config=config)
rprint(state)

### 关键步骤

1. 初始化记忆引擎：checkpointer = InMemorySaver() -- 创建一个内存级的记忆存储  

    InMemorySaver在内存中保存，进程结束就丢失数据。生产环境中需要换成数据库持久化的 SqliteSaver、PostgresSaver 等

2. 绑定 Agent：在 create_agent 时传入 checkpointer，让 Agent 具备状态存储能力。

3. 设置会话ID：通过 config = {"configurable":{"thread_id" = "xx"}} 为每次调用指定线程标识。**同一个 thread_id 共享记忆，不同 thread_id 完全隔离***

### 工作原理

在追问时，checkpointer 会自动：读取历史消息 -> 追加新消息 -> 调用模型（传入完整历史）-> 保存新历史

## 2.2 基于外部存储介质的持久化

如果将状态检查点（checkpoint）保存在内存，进程终止时丢失会发生，生命周期不可接受。因此，生命周期使用持久化的外部储存介质，如 PostgreSQL。LangGraph 提供的 checkpoint 后端表如下：

https://docs.langchain.com/oss/python/langgraph/persistence#checkpoint-libraries

此处选用 PostgreSQL 作为持久化器。

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.postgres import PostgresSaver

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

DB_URL = os.getenv("DATABASE_URL")

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    print("已连接 PostgreSQL")
    checkpointer.setup()
    print("checkpoint 表已就绪")

    agent = create_agent(model=model, checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "1"}}

    print("开始调用模型")
    resp = agent.invoke(
        {"messages": [HumanMessage("你好，我是艾海鹏，喜欢仓鼠")]},
        config=config,
    )

    resp = agent.invoke(
        {"messages": [HumanMessage("还记的我的名字和喜欢的小动物么？")]},
        config=config,
    )

    for msg in resp["messages"]:
        msg.pretty_print()

## 2.3 二者的区别

基于内存的短期记忆（InMemorySaver）和基于数据库的短期记忆（PostgresSaver）对比：

| 维度 | InMemorySaver（内存） | PostgresSaver（数据库） |
| --- | --- | --- |
| 存储位置 | 进程内存 | 外部数据库 |
| 生命周期 | 进程结束/重启即丢失 | 跨进程、跨机器重启存活 |
| 共享 | 仅当前进程内可见 | 多实例/多进程共享同一份历史 |
| 性能 | 最快（无 IO） | 有网络/磁盘 IO |
| 适用场景 | 演示、原型、单进程 | 生产服务、多实例部署、需要审计 |

**核心区别**：两者都是会话级记忆（换 `thread_id` 即消失），数据库方案解决的不是"记忆更长"，而是**可靠性**——进程崩溃、重启、水平扩展时历史不丢，所有实例看到同一份状态。

**示例**：Postgres 场景下启动 3 个 agent 实例，实例 A 写的对话，实例 B 用同一 `thread_id` 能接上；内存场景做不到。